# Kronos-small development comparison

This is an **exploratory** zero-shot comparison, not Phase 8 or a blind test. The 10 tickers, 3 dates, 90-session lookback, and 20-session horizon below are fixed before running. We use the frozen Phase 6 development dataset, not the Phase 8 cleaned snapshot. Raw OHLCV forecasts are compared with raw-price outcomes and raw trailing-price controls. These results cannot be presented as the project's adjusted-price Phase 8 scores. No model is trained here.


In [1]:
from pathlib import Path
import os, subprocess, sys, zipfile

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    root = Path('/content/HalalifyAI')
    if not root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/frager098/HalalifyAI.git', str(root)], check=True)
    os.chdir(root)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch>=2.0', 'einops==0.8.1', 'huggingface_hub==0.33.1', 'safetensors==0.6.2', 'pandas', 'numpy'], check=True)
else:
    root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'AGENTS.md').is_file() and (p/'src').is_dir() and (p/'configs').is_dir()), None)
    if root is None:
        raise RuntimeError('Open this notebook from inside the HalalifyAI repository.')
    os.chdir(root)
print('Environment:', 'Colab' if IN_COLAB else 'local')
print('Repository root:', root)
print('Notebook kernel Python:', sys.executable)


Environment: local
Repository root: d:\Arham\FYP\HalalifyAI
Notebook kernel Python: d:\Arham\FYP\HalalifyAI\.venv-phases\Scripts\python.exe


In [2]:
import hashlib, importlib.util, json
contract = json.loads(Path('configs/phase8_evaluation_contract_v1.json').read_text(encoding='utf-8'))
source_info = contract['input_identities']['development_dataset']
dataset_path = Path(source_info['path'])
print('Required input:', dataset_path)
if IN_COLAB and not dataset_path.is_file():
    from google.colab import files
    print('Upload kronos_development_inputs.zip with only the printed repository-relative CSV path.')
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
        raise ValueError('Upload exactly one ZIP file.')
    with zipfile.ZipFile(next(iter(uploaded))) as bundle:
        members = bundle.infolist()
        expected = dataset_path.as_posix()
        if [m.filename for m in members if not m.is_dir()] != [expected]:
            raise ValueError(f'ZIP must contain only {expected} at that exact path.')
        for member in members:
            target = (root / member.filename).resolve()
            if not target.is_relative_to(root.resolve()) or (target.exists() and not member.is_dir()):
                raise ValueError(f'Unsafe or existing ZIP target: {member.filename}')
        bundle.extractall(root)
if not dataset_path.is_file():
    raise FileNotFoundError(f'Missing {dataset_path}. Place the accepted development CSV at this exact path.')
dataset_hash = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
if dataset_hash != source_info['sha256']:
    raise ValueError(f'Development CSV SHA-256 mismatch: {dataset_hash}')
required = ['numpy', 'pandas', 'torch', 'einops', 'huggingface_hub', 'safetensors']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'Missing {missing} in notebook kernel {sys.executable}. Select the environment with Kronos dependencies.')
print('Input hash and kernel dependencies verified.')


Required input: data\processed\model_inputs\phase6_expanded_features_v1\market_features_and_labels.csv
Input hash and kernel dependencies verified.


## Fixed exploration plan

Each date uses the same ten tickers. The 20 future observed session dates form a calendar only; future prices are not passed to Kronos. Return ranking is measured with within-date Spearman across the ten stocks. Risk is measured as the absolute error of predicted annualized 20-session realized volatility. Both are compared with raw trailing-price controls. Three dates and ten selected stocks remain a small, non-blind sample; do not change them after viewing results and then call the result a fresh test.


In [3]:
import numpy as np
import pandas as pd
TICKERS = ('AAPL', 'MSFT', 'NVDA', 'AMZN', 'JNJ', 'XOM', 'JPM', 'WMT', 'CAT', 'PG')
DATES = tuple(pd.Timestamp(x) for x in ('2024-09-03', '2025-03-03', '2025-09-02'))
LOOKBACK, HORIZON, SEED = 90, 20, 20261127
columns = ['ticker', 'session_date', 'open', 'high', 'low', 'close', 'volume', 'target_window_end', 'return_eligible']
frame = pd.read_csv(dataset_path, usecols=columns, parse_dates=['session_date', 'target_window_end'])
stock_rows = {ticker: frame.loc[frame.ticker.eq(ticker)].sort_values('session_date').reset_index(drop=True) for ticker in TICKERS}
cases = []
for date in DATES:
    for ticker in TICKERS:
        table = stock_rows[ticker]
        if table.session_date.duplicated().any():
            raise ValueError(f'Duplicate dates for {ticker}')
        hit = table.index[table.session_date.eq(date)]
        if len(hit) != 1:
            raise ValueError(f'{ticker} missing decision date {date.date()}')
        i = int(hit[0])
        if i < LOOKBACK or len(table) - i - 1 < HORIZON:
            raise ValueError(f'Insufficient bars for {ticker} {date.date()}')
        history = table.iloc[i-LOOKBACK+1:i+1].copy()
        future = table.iloc[i+1:i+1+HORIZON].copy()
        if table.loc[i, 'return_eligible'] != True or table.loc[i, 'target_window_end'] != future.session_date.iloc[-1]:
            raise ValueError(f'Eligibility or target-window mismatch for {ticker} {date.date()}')
        if not np.isfinite(history[['open','high','low','close','volume']].to_numpy(dtype=float)).all() or not np.isfinite(future.close.to_numpy(dtype=float)).all():
            raise ValueError(f'Nonfinite bars for {ticker} {date.date()}')
        cases.append((ticker, date, history, future))
print('Frozen cases:', len(cases), 'across', len(DATES), 'dates and', len(TICKERS), 'stocks.')
print('First/last target ends:', cases[0][3].session_date.iloc[-1].date(), cases[-1][3].session_date.iloc[-1].date())


Frozen cases: 30 across 3 dates and 10 stocks.
First/last target ends: 2024-10-01 2025-09-30


In [4]:
UPSTREAM_COMMIT = '67b630e67f6a18c9e9be918d9b4337c960db1e9a'
MODEL_REVISION = '901c26c1332695a2a8f243eb2f37243a37bea320'
TOKENIZER_REVISION = '0e0117387f39004a9016484a186a908917e22426'
upstream = Path('data/interim/kronos_upstream').resolve()
if not upstream.exists():
    subprocess.run(['git', 'clone', 'https://github.com/shiyu-coder/Kronos.git', str(upstream)], check=True)
    subprocess.run(['git', '-C', str(upstream), 'checkout', '--detach', UPSTREAM_COMMIT], check=True)
commit = subprocess.check_output(['git', '-C', str(upstream), 'rev-parse', 'HEAD'], text=True).strip()
if commit != UPSTREAM_COMMIT:
    raise RuntimeError(f'Existing Kronos clone is {commit}; this notebook requires {UPSTREAM_COMMIT}.')
sys.path.insert(0, str(upstream))
import random, torch
from model import Kronos, KronosTokenizer, KronosPredictor
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
tokenizer = KronosTokenizer.from_pretrained('NeoQuasar/Kronos-Tokenizer-base', revision=TOKENIZER_REVISION)
model = Kronos.from_pretrained('NeoQuasar/Kronos-small', revision=MODEL_REVISION)
predictor = KronosPredictor(model, tokenizer, max_context=512)
print('Loaded Kronos-small; code commit:', commit, 'device:', 'cuda' if torch.cuda.is_available() else 'cpu')


d:\Arham\FYP\HalalifyAI\.venv-phases\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded Kronos-small; code commit: 67b630e67f6a18c9e9be918d9b4337c960db1e9a device: cpu


## Generate the 30 forecasts

This can take time on a CPU. The cell prints progress. If it is interrupted, rerun the cell from its beginning; no result file is written until the final cell. The fixed seed and case order are reset here.


In [5]:
if 'predictor' not in globals():
    raise RuntimeError('Run the model-loading cell successfully before forecasting.')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
records = []
for number, (ticker, date, history, future) in enumerate(cases, start=1):
    x = history[['open','high','low','close','volume']].reset_index(drop=True)
    x_dates = history.session_date.reset_index(drop=True)
    y_dates = future.session_date.reset_index(drop=True)
    predicted = predictor.predict(df=x, x_timestamp=x_dates, y_timestamp=y_dates, pred_len=HORIZON, T=1.0, top_p=0.9, sample_count=1)
    predicted_close = predicted.close.to_numpy(dtype=float)
    actual_close = future.close.to_numpy(dtype=float)
    if len(predicted_close) != HORIZON or not np.isfinite(predicted_close).all():
        raise ValueError(f'Invalid Kronos forecast for {ticker} {date.date()}')
    start_close = float(history.close.iloc[-1])
    # Include the known close at t so the 20 one-day returns cover t+1 through t+20.
    predicted_daily = np.diff(np.r_[start_close, predicted_close]) / np.r_[start_close, predicted_close][:-1]
    actual_daily = np.diff(np.r_[start_close, actual_close]) / np.r_[start_close, actual_close][:-1]
    trailing_close = history.close.to_numpy(dtype=float)
    trailing_daily = np.diff(trailing_close[-21:]) / trailing_close[-21:-1]
    records.append({'ticker': ticker, 'decision_date': str(date.date()), 'target_end': str(future.session_date.iloc[-1].date()),
                    'predicted_raw_return': float(predicted_close[-1] / start_close - 1),
                    'actual_raw_return': float(actual_close[-1] / start_close - 1),
                    'trailing_raw_momentum_20d': float(start_close / trailing_close[-21] - 1),
                    'predicted_raw_volatility': float(np.std(predicted_daily, ddof=1) * np.sqrt(252)),
                    'actual_raw_volatility': float(np.std(actual_daily, ddof=1) * np.sqrt(252)),
                    'trailing_raw_volatility_20d': float(np.std(trailing_daily, ddof=1) * np.sqrt(252))})
    print(f'{number}/{len(cases)} {ticker} {date.date()} complete', flush=True)
predictions = pd.DataFrame.from_records(records)
display(predictions.head())


100%|██████████| 20/20 [00:02<00:00,  7.99it/s]

1/30 AAPL 2024-09-03 complete



100%|██████████| 20/20 [00:03<00:00,  6.03it/s]

2/30 MSFT 2024-09-03 complete



100%|██████████| 20/20 [00:02<00:00,  7.80it/s]

3/30 NVDA 2024-09-03 complete


... (51 outputs left out)

29/30 CAT 2025-09-02 complete



100%|██████████| 20/20 [00:03<00:00,  6.10it/s]


30/30 PG 2025-09-02 complete


,ticker,decision_date,target_end,predicted_raw_return,actual_raw_return,trailing_raw_momentum_20d,predicted_raw_volatility,actual_raw_volatility,trailing_raw_volatility_20d
0,AAPL,2024-09-03,2024-10-01,-0.117535,0.015442,0.064510,0.298846,0.238253,0.173713
1,MSFT,2024-09-03,2024-10-01,0.095397,0.027477,0.036163,0.160942,0.189588,0.158183
2,NVDA,2024-09-03,2024-10-01,0.127334,0.083333,0.075162,1.003042,0.470310,0.689613
3,AMZN,2024-09-03,2024-10-01,0.065393,0.050383,0.094585,0.153938,0.269837,0.258602
4,JNJ,2024-09-03,2024-10-01,-0.123061,-0.030928,0.036651,0.204102,0.120082,0.107648


In [6]:
if len(predictions) != len(cases):
    raise RuntimeError('Not all forecasts completed.')
summary_rows = []
for date, group in predictions.groupby('decision_date', sort=True):
    summary_rows.append({'decision_date': date, 'stocks': len(group),
        'kronos_return_spearman': group.predicted_raw_return.corr(group.actual_raw_return, method='spearman'),
        'trailing_momentum_spearman': group.trailing_raw_momentum_20d.corr(group.actual_raw_return, method='spearman'),
        'kronos_volatility_mae': float((group.predicted_raw_volatility - group.actual_raw_volatility).abs().mean()),
        'trailing_volatility_mae': float((group.trailing_raw_volatility_20d - group.actual_raw_volatility).abs().mean())})
summary = pd.DataFrame(summary_rows)
print('Development-only raw-price comparison (smaller volatility MAE is better):')
display(summary)
print('Mean Kronos/Control return Spearman:', summary[['kronos_return_spearman','trailing_momentum_spearman']].mean().to_dict())
print('Mean Kronos/Control volatility MAE:', summary[['kronos_volatility_mae','trailing_volatility_mae']].mean().to_dict())
print('These 30 cases are not the full 50-stock project evaluation, adjusted-close Phase 8 result, or an untouched test.')


Development-only raw-price comparison (smaller volatility MAE is better):


,decision_date,stocks,kronos_return_spearman,trailing_momentum_spearman,kronos_volatility_mae,trailing_volatility_mae
0,2024-09-03,10,0.660606,0.236364,0.129923,0.059386
1,2025-03-03,10,0.187879,0.309091,0.061526,0.050301
2,2025-09-02,10,-0.660606,-0.296970,0.069951,0.054738


Mean Kronos/Control return Spearman: {'kronos_return_spearman': 0.06262626262626263, 'trailing_momentum_spearman': 0.08282828282828282}
Mean Kronos/Control volatility MAE: {'kronos_volatility_mae': 0.08713354295781506, 'trailing_volatility_mae': 0.054808312386606106}
These 30 cases are not the full 50-stock project evaluation, adjusted-close Phase 8 result, or an untouched test.


In [7]:
import importlib.metadata as metadata
out_dir = Path('reports/model_evaluation/kronos_small_development_v1')
out_dir.mkdir(parents=True, exist_ok=True)
detail_path = out_dir / 'forecast_cases.csv'
summary_path = out_dir / 'comparison_summary.csv'
metadata_path = out_dir / 'run_metadata.json'
for path in (detail_path, summary_path, metadata_path):
    if path.exists():
        raise FileExistsError(f'Existing result preserved: {path}')
run_info = {'status': 'exploratory_development_only', 'dataset_path': str(dataset_path), 'dataset_sha256': dataset_hash,
    'tickers': list(TICKERS), 'decision_dates': [str(d.date()) for d in DATES], 'lookback': LOOKBACK, 'horizon': HORIZON,
    'upstream_commit': commit, 'model_revision': MODEL_REVISION, 'tokenizer_revision': TOKENIZER_REVISION,
    'seed': SEED, 'torch_version': torch.__version__, 'huggingface_hub_version': metadata.version('huggingface_hub'),
    'sampling': {'T': 1.0, 'top_p': 0.9, 'sample_count': 1},
    'interpretation': 'Raw-price, selected-stock development comparison. Not Phase 8, not blind, no model fitting, no Shariah inference.'}
predictions.to_csv(detail_path, index=False)
summary.to_csv(summary_path, index=False)
metadata_path.write_text(json.dumps(run_info, indent=2) + '\n', encoding='utf-8')
print('Created:', detail_path, summary_path, metadata_path, sep='\n')


Created:
reports\model_evaluation\kronos_small_development_v1\forecast_cases.csv
reports\model_evaluation\kronos_small_development_v1\comparison_summary.csv
reports\model_evaluation\kronos_small_development_v1\run_metadata.json
